# 11. Modelos reales por lote

Evalúa GLM, MLP y PINN inversa Balloon–Windkessel en ocho experimentos reales:

- cuatro escenarios;
- entrenamiento bloque 1 → evaluación bloque 2;
- entrenamiento bloque 2 → evaluación bloque 1.

La corrección local de línea base utiliza la **mediana preestímulo**. El bloque de evaluación no se desplaza temporalmente.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip install -q nilearn

In [ ]:
from pathlib import Path
import copy
import json
import random
import time
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn

from scipy.integrate import solve_ivp
from nilearn.glm.first_level import make_first_level_design_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
PROJECT_ROOT = Path("/content/drive/MyDrive/Proyecto_PINN_HRF")
SUBJECT_ID = "100206"

REAL_MODEL_DIR = (
    PROJECT_ROOT / "data" / "processed" / SUBJECT_ID / "real_model_inputs"
)

RAW_RESULTS_DIR = (
    PROJECT_ROOT / "data" / "raw" / SUBJECT_ID / "MNINonLinear" / "Results"
)

RESULTS_DIR = (
    PROJECT_ROOT / "results" / "real" / SUBJECT_ID / "models_batch"
)

PREDICTIONS_DIR = RESULTS_DIR / "predictions"
HISTORIES_DIR = RESULTS_DIR / "histories"
HRF_DIR = RESULTS_DIR / "hrf_estimates"
MODELS_DIR = RESULTS_DIR / "models"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"

for directory in [
    RESULTS_DIR,
    PREDICTIONS_DIR,
    HISTORIES_DIR,
    HRF_DIR,
    MODELS_DIR,
    FIGURES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

METRICS_PATH = RESULTS_DIR / "real_model_metrics.csv"
PARAMETERS_PATH = RESULTS_DIR / "real_pinn_parameters.csv"
STATUS_PATH = RESULTS_DIR / "real_experiment_status.csv"
FAILURES_PATH = RESULTS_DIR / "real_experiment_failures.csv"

print("Directorio de resultados:", RESULTS_DIR)

In [ ]:
SCENARIOS = {
    "motor_lr_m1_left": {
        "run": "MOTOR_LR",
        "roi": "M1_left",
        "target_condition": "rh",
        "full_signal_path": REAL_MODEL_DIR / "motor_lr_m1_left_full.csv",
        "event_file": RAW_RESULTS_DIR / "tfMRI_MOTOR_LR" / "EVs" / "rh.txt",
    },
    "motor_lr_m1_right": {
        "run": "MOTOR_LR",
        "roi": "M1_right",
        "target_condition": "lh",
        "full_signal_path": REAL_MODEL_DIR / "motor_lr_m1_right_full.csv",
        "event_file": RAW_RESULTS_DIR / "tfMRI_MOTOR_LR" / "EVs" / "lh.txt",
    },
    "motor_rl_m1_left": {
        "run": "MOTOR_RL",
        "roi": "M1_left",
        "target_condition": "rh",
        "full_signal_path": REAL_MODEL_DIR / "motor_rl_m1_left_full.csv",
        "event_file": RAW_RESULTS_DIR / "tfMRI_MOTOR_RL" / "EVs" / "rh.txt",
    },
    "motor_rl_m1_right": {
        "run": "MOTOR_RL",
        "roi": "M1_right",
        "target_condition": "lh",
        "full_signal_path": REAL_MODEL_DIR / "motor_rl_m1_right_full.csv",
        "event_file": RAW_RESULTS_DIR / "tfMRI_MOTOR_RL" / "EVs" / "lh.txt",
    },
}

for scenario_name, config in SCENARIOS.items():
    print(
        scenario_name,
        "señal:", config["full_signal_path"].exists(),
        "| eventos:", config["event_file"].exists(),
    )

EXPERIMENTS = []

for scenario_index, (scenario_name, config) in enumerate(SCENARIOS.items()):
    for train_block, test_block in [(1, 2), (2, 1)]:
        EXPERIMENTS.append(
            {
                "scenario": scenario_name,
                "run": config["run"],
                "roi": config["roi"],
                "target_condition": config["target_condition"],
                "train_block": train_block,
                "test_block": test_block,
                "experiment_seed": 20260717 + 100 * scenario_index + 10 * train_block + test_block,
            }
        )

experiment_table = pd.DataFrame(EXPERIMENTS)
display(experiment_table)
print("Experimentos:", len(experiment_table))

In [ ]:
SECONDS_BEFORE = 8.0
SECONDS_AFTER = 20.0
EVALUATION_START_S = 0.0
EVALUATION_END_S = 24.0

MLP_MAXIMUM_LAG_S = 24.0

PINN_CONFIG = {
    "hidden_layers": [64, 64, 64],
    "phase_1_epochs": 1200,
    "phase_2_epochs": 3500,
    "phase_1_learning_rate": 1e-3,
    "phase_2_network_learning_rate": 3e-4,
    "phase_2_parameter_learning_rate": 8e-4,
    "lambda_physics_phase_1": 5.0,
    "lambda_physics_phase_2": 20.0,
    "lambda_initial": 100.0,
    "lambda_terminal": 5.0,
    "n_collocation": 700,
    "residual_scale": 0.05,
    "active_data_weight": 3.0,
    "lbfgs_max_iterations": 300,
}

INITIAL_PARAMETERS = {
    "epsilon": 0.12,
    "tau": 1.20,
    "alpha": 0.38,
}

PARAMETER_BOUNDS = {
    "epsilon": (0.03, 0.35),
    "tau": (0.50, 2.00),
    "alpha": (0.15, 0.60),
}

torch.set_default_dtype(torch.float64)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Dispositivo:", DEVICE)
print("Precisión:", torch.get_default_dtype())

In [ ]:
def establecer_semilla(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(True, warn_only=True)


def correlacion_segura(
    observed: np.ndarray,
    predicted: np.ndarray,
) -> float:
    observed = np.asarray(observed, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    if np.std(observed) < 1e-12 or np.std(predicted) < 1e-12:
        return np.nan

    return float(np.corrcoef(observed, predicted)[0, 1])


def calcular_metricas(
    observed: np.ndarray,
    predicted: np.ndarray,
) -> dict:
    observed = np.asarray(observed, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    mse = mean_squared_error(observed, predicted)

    return {
        "rmse_percent": float(100.0 * np.sqrt(mse)),
        "mae_percent": float(100.0 * mean_absolute_error(observed, predicted)),
        "r2": float(r2_score(observed, predicted)),
        "pearson_r": correlacion_segura(observed, predicted),
    }

In [ ]:
def extraer_ventana_real(
    frame_times: np.ndarray,
    signal: np.ndarray,
    event: pd.Series,
    seconds_before: float,
    seconds_after: float,
) -> dict:
    onset_s = float(event["onset_s"])
    duration_s = float(event["duration_s"])

    start_s = max(float(frame_times[0]), onset_s - seconds_before)
    end_s = min(
        float(frame_times[-1]),
        onset_s + duration_s + seconds_after,
    )

    mask = (frame_times >= start_s) & (frame_times <= end_s)

    absolute_time = frame_times[mask]
    relative_time = absolute_time - onset_s
    model_time = relative_time + seconds_before
    observed_signal = signal[mask]

    prestimulus_mask = relative_time < 0.0

    if prestimulus_mask.sum() < 5:
        raise RuntimeError("La ventana contiene pocos puntos preestímulo.")

    prestimulus_signal = observed_signal[prestimulus_mask]
    baseline_value = float(np.median(prestimulus_signal))

    baseline = np.full_like(
        observed_signal,
        baseline_value,
        dtype=float,
    )

    response = observed_signal - baseline_value

    diagnostic_coefficients = np.polyfit(
        relative_time[prestimulus_mask],
        prestimulus_signal,
        deg=1,
    )

    stimulus = (
        (relative_time >= 0.0)
        & (relative_time < duration_s)
    ).astype(float)

    evaluation_mask = (
        (relative_time >= EVALUATION_START_S)
        & (relative_time <= EVALUATION_END_S)
    )

    return {
        "absolute_time_s": absolute_time,
        "relative_time_s": relative_time,
        "model_time_s": model_time,
        "observed_fraction": observed_signal,
        "baseline_fraction": baseline,
        "response_fraction": response,
        "stimulus": stimulus,
        "prestimulus_mask": prestimulus_mask,
        "evaluation_mask": evaluation_mask,
        "baseline_slope": float(diagnostic_coefficients[0]),
        "baseline_intercept": baseline_value,
        "onset_s": onset_s,
        "duration_s": duration_s,
    }


def cargar_contexto_experimento(
    scenario_name: str,
    train_block: int,
    test_block: int,
) -> dict:
    config = SCENARIOS[scenario_name]

    full_table = pd.read_csv(config["full_signal_path"])

    event_values = np.loadtxt(
        config["event_file"],
        ndmin=2,
    )

    events = pd.DataFrame(
        {
            "onset_s": event_values[:, 0],
            "duration_s": event_values[:, 1],
            "amplitude": event_values[:, 2],
        }
    ).sort_values("onset_s").reset_index(drop=True)

    if len(events) != 2:
        raise ValueError(
            f"{scenario_name}: se esperaban dos bloques y se encontraron {len(events)}."
        )

    time_full = full_table["time_s"].to_numpy(dtype=float)
    bold_adjusted = full_table["bold_adjusted_fraction"].to_numpy(dtype=float)

    tr_s = float(np.median(np.diff(time_full)))

    windows = {}

    for block_index, event in events.iterrows():
        block_number = block_index + 1

        windows[block_number] = extraer_ventana_real(
            frame_times=time_full,
            signal=bold_adjusted,
            event=event,
            seconds_before=SECONDS_BEFORE,
            seconds_after=SECONDS_AFTER,
        )

    return {
        "config": config,
        "events": events,
        "time_full": time_full,
        "bold_adjusted": bold_adjusted,
        "tr_s": tr_s,
        "train_window": windows[train_block],
        "test_window": windows[test_block],
        "train_block": train_block,
        "test_block": test_block,
    }

In [ ]:
def crear_regresor_glm(
    model_times: np.ndarray,
    stimulus_onset_s: float,
    stimulus_duration_s: float,
) -> np.ndarray:
    events_glm = pd.DataFrame(
        {
            "onset": [stimulus_onset_s],
            "duration": [stimulus_duration_s],
            "trial_type": ["stimulus"],
        }
    )

    design = make_first_level_design_matrix(
        frame_times=np.asarray(model_times, dtype=float),
        events=events_glm,
        hrf_model="spm",
        drift_model=None,
        min_onset=-24,
    )

    return design["stimulus"].to_numpy(dtype=float)


def ejecutar_glm(
    train_window: dict,
    test_window: dict,
) -> dict:
    stimulus_onset_model_s = SECONDS_BEFORE
    stimulus_duration_s = float(train_window["duration_s"])

    train_regressor = crear_regresor_glm(
        model_times=train_window["model_time_s"],
        stimulus_onset_s=stimulus_onset_model_s,
        stimulus_duration_s=stimulus_duration_s,
    )

    test_regressor = crear_regresor_glm(
        model_times=test_window["model_time_s"],
        stimulus_onset_s=stimulus_onset_model_s,
        stimulus_duration_s=stimulus_duration_s,
    )

    denominator = float(np.dot(train_regressor, train_regressor))

    if denominator < 1e-12:
        raise RuntimeError("El regresor GLM tiene variabilidad insuficiente.")

    beta = float(
        np.dot(
            train_regressor,
            train_window["response_fraction"],
        )
        / denominator
    )

    return {
        "beta_fraction": beta,
        "train_prediction": beta * train_regressor,
        "test_prediction": beta * test_regressor,
    }

In [ ]:
def construir_retardos(
    stimulus: np.ndarray,
    n_lags: int,
) -> np.ndarray:
    stimulus = np.asarray(stimulus, dtype=float)

    matrix = np.zeros(
        (len(stimulus), n_lags),
        dtype=float,
    )

    for lag in range(n_lags):
        if lag == 0:
            matrix[:, lag] = stimulus
        else:
            matrix[lag:, lag] = stimulus[:-lag]

    return matrix


class RealStimulusMLP(nn.Module):
    def __init__(self, input_dimension: int):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dimension, 32),
            nn.Tanh(),
            nn.Linear(32, 16),
            nn.Tanh(),
            nn.Linear(16, 1),
        )

    def forward(self, inputs: torch.Tensor):
        return self.network(inputs)


def entrenar_mlp_real(
    x_train: np.ndarray,
    y_train_percent: np.ndarray,
    relative_time_s: np.ndarray,
    random_seed: int,
) -> dict:
    establecer_semilla(random_seed)

    n_points = len(x_train)
    rng = np.random.default_rng(random_seed)
    shuffled_indices = rng.permutation(n_points)

    n_validation = max(
        5,
        int(round(0.20 * n_points)),
    )

    validation_indices = shuffled_indices[:n_validation]
    fitting_indices = shuffled_indices[n_validation:]

    x_mean = np.mean(x_train, axis=0, keepdims=True)
    x_std = np.std(x_train, axis=0, ddof=0, keepdims=True)
    x_std = np.where(x_std < 1e-8, 1.0, x_std)

    y_mean = float(np.mean(y_train_percent))
    y_std = float(np.std(y_train_percent, ddof=0))

    if y_std < 1e-8:
        y_std = 1.0

    x_scaled = (x_train - x_mean) / x_std
    y_scaled = (y_train_percent - y_mean) / y_std

    sample_weights = np.ones(n_points, dtype=float)

    active_mask = (
        (relative_time_s >= 0.0)
        & (relative_time_s <= EVALUATION_END_S)
    )

    sample_weights[active_mask] = 3.0

    x_tensor = torch.tensor(x_scaled, device=DEVICE)
    y_tensor = torch.tensor(y_scaled[:, None], device=DEVICE)
    weights_tensor = torch.tensor(sample_weights[:, None], device=DEVICE)

    fitting_tensor = torch.tensor(
        fitting_indices,
        dtype=torch.long,
        device=DEVICE,
    )

    validation_tensor = torch.tensor(
        validation_indices,
        dtype=torch.long,
        device=DEVICE,
    )

    model = RealStimulusMLP(
        input_dimension=x_train.shape[1]
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-3,
    )

    best_validation_loss = np.inf
    best_state = None
    best_epoch = 0
    epochs_without_improvement = 0

    maximum_epochs = 2500
    patience = 200
    history = []

    for epoch in range(1, maximum_epochs + 1):
        model.train()
        optimizer.zero_grad()

        fitting_prediction = model(x_tensor[fitting_tensor])

        fitting_error = (
            fitting_prediction
            - y_tensor[fitting_tensor]
        ) ** 2

        fitting_loss = torch.sum(
            weights_tensor[fitting_tensor] * fitting_error
        ) / torch.sum(weights_tensor[fitting_tensor])

        fitting_loss.backward()
        optimizer.step()

        model.eval()

        with torch.no_grad():
            validation_prediction = model(
                x_tensor[validation_tensor]
            )

            validation_loss = torch.mean(
                (
                    validation_prediction
                    - y_tensor[validation_tensor]
                ) ** 2
            )

        validation_value = float(validation_loss.cpu())

        history.append(
            {
                "epoch": epoch,
                "training_loss": float(
                    fitting_loss.detach().cpu()
                ),
                "validation_loss": validation_value,
            }
        )

        if validation_value < best_validation_loss - 1e-7:
            best_validation_loss = validation_value
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    if best_state is None:
        raise RuntimeError("La MLP no generó un estado válido.")

    model.load_state_dict(best_state)

    return {
        "model": model,
        "x_mean": x_mean,
        "x_std": x_std,
        "y_mean": y_mean,
        "y_std": y_std,
        "best_epoch": int(best_epoch),
        "best_validation_loss": float(best_validation_loss),
        "history": pd.DataFrame(history),
    }


def predecir_mlp_real(
    training_result: dict,
    input_matrix: np.ndarray,
    prestimulus_mask: np.ndarray,
) -> np.ndarray:
    input_scaled = (
        input_matrix - training_result["x_mean"]
    ) / training_result["x_std"]

    input_tensor = torch.tensor(
        input_scaled,
        device=DEVICE,
    )

    training_result["model"].eval()

    with torch.no_grad():
        prediction_scaled = (
            training_result["model"](input_tensor)
            .cpu()
            .numpy()
            .reshape(-1)
        )

    prediction_percent = (
        prediction_scaled
        * training_result["y_std"]
        + training_result["y_mean"]
    )

    prediction_percent = (
        prediction_percent
        - np.mean(prediction_percent[prestimulus_mask])
    )

    return prediction_percent / 100.0


def ejecutar_mlp(
    train_window: dict,
    test_window: dict,
    tr_s: float,
    random_seed: int,
) -> dict:
    n_lags = int(round(MLP_MAXIMUM_LAG_S / tr_s)) + 1

    x_train = construir_retardos(
        train_window["stimulus"],
        n_lags,
    )

    x_test = construir_retardos(
        test_window["stimulus"],
        n_lags,
    )

    y_train_percent = (
        100.0 * train_window["response_fraction"]
    )

    training_result = entrenar_mlp_real(
        x_train=x_train,
        y_train_percent=y_train_percent,
        relative_time_s=train_window["relative_time_s"],
        random_seed=random_seed,
    )

    train_prediction = predecir_mlp_real(
        training_result=training_result,
        input_matrix=x_train,
        prestimulus_mask=train_window["prestimulus_mask"],
    )

    test_prediction = predecir_mlp_real(
        training_result=training_result,
        input_matrix=x_test,
        prestimulus_mask=test_window["prestimulus_mask"],
    )

    return {
        "training_result": training_result,
        "train_prediction": train_prediction,
        "test_prediction": test_prediction,
        "n_lags": n_lags,
    }

In [ ]:
def logit(probability: float) -> float:
    probability = np.clip(
        probability,
        1e-8,
        1.0 - 1e-8,
    )

    return float(
        np.log(
            probability
            / (1.0 - probability)
        )
    )


class BoundedScalar(nn.Module):
    def __init__(
        self,
        lower: float,
        upper: float,
        initial: float,
    ):
        super().__init__()

        self.lower = float(lower)
        self.upper = float(upper)

        normalized = (
            (initial - lower)
            / (upper - lower)
        )

        self.raw = nn.Parameter(
            torch.tensor(logit(normalized))
        )

    def forward(self):
        return (
            self.lower
            + (self.upper - self.lower)
            * torch.sigmoid(self.raw)
        )


class RealWindowPINN(nn.Module):
    def __init__(self, maximum_time_s: float):
        super().__init__()

        self.maximum_time_s = float(maximum_time_s)

        dimensions = [
            1,
            *PINN_CONFIG["hidden_layers"],
            4,
        ]

        layers = []

        for input_size, output_size in zip(
            dimensions[:-2],
            dimensions[1:-1],
        ):
            layers.append(
                nn.Linear(input_size, output_size)
            )
            layers.append(nn.Tanh())

        layers.append(
            nn.Linear(
                dimensions[-2],
                dimensions[-1],
            )
        )

        self.network = nn.Sequential(*layers)

        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

        self.epsilon = BoundedScalar(
            *PARAMETER_BOUNDS["epsilon"],
            INITIAL_PARAMETERS["epsilon"],
        )

        self.tau = BoundedScalar(
            *PARAMETER_BOUNDS["tau"],
            INITIAL_PARAMETERS["tau"],
        )

        self.alpha = BoundedScalar(
            *PARAMETER_BOUNDS["alpha"],
            INITIAL_PARAMETERS["alpha"],
        )

    def states(self, time_tensor: torch.Tensor):
        normalized_time = (
            2.0
            * time_tensor
            / self.maximum_time_s
            - 1.0
        )

        raw = self.network(normalized_time)

        s = 0.50 * torch.tanh(raw[:, 0:1])
        f = 1.0 + 0.80 * torch.tanh(raw[:, 1:2])
        v = 1.0 + 0.35 * torch.tanh(raw[:, 2:3])
        q = 1.0 + 0.35 * torch.tanh(raw[:, 3:4])

        return s, f, v, q

    def parameter_values(self):
        return {
            "epsilon": self.epsilon(),
            "tau": self.tau(),
            "alpha": self.alpha(),
        }

    def bold(self, time_tensor: torch.Tensor):
        _, _, v, q = self.states(time_tensor)

        E0 = 0.34
        V0 = 0.02
        k1 = 7.0 * E0
        k2 = 2.0
        k3 = 2.0 * E0 - 0.2

        return V0 * (
            k1 * (1.0 - q)
            + k2 * (1.0 - q / v)
            + k3 * (1.0 - v)
        )

In [ ]:
PINN_STIMULUS_ONSET = None
PINN_STIMULUS_END = None
pinn_observed_time = None
pinn_observed_bold = None
pinn_data_scale = None
pinn_weight_tensor = None
collocation_tensor = None
PINN_TERMINAL_TIME = None


def preparar_contexto_pinn(
    train_window: dict,
) -> None:
    global PINN_STIMULUS_ONSET
    global PINN_STIMULUS_END
    global pinn_observed_time
    global pinn_observed_bold
    global pinn_data_scale
    global pinn_weight_tensor
    global collocation_tensor
    global PINN_TERMINAL_TIME

    PINN_STIMULUS_ONSET = SECONDS_BEFORE
    PINN_STIMULUS_END = (
        SECONDS_BEFORE
        + float(train_window["duration_s"])
    )

    pinn_observed_time = torch.tensor(
        train_window["model_time_s"][:, None],
        device=DEVICE,
    )

    pinn_observed_bold = torch.tensor(
        train_window["response_fraction"][:, None],
        device=DEVICE,
    )

    pinn_data_scale = float(
        np.std(
            train_window["response_fraction"],
            ddof=0,
        )
    )

    if pinn_data_scale < 1e-8:
        raise ValueError(
            "La señal de entrenamiento tiene variabilidad insuficiente."
        )

    pinn_weights = np.ones(
        len(train_window["model_time_s"]),
        dtype=float,
    )

    pinn_active_mask = (
        (train_window["relative_time_s"] >= 0.0)
        & (
            train_window["relative_time_s"]
            <= EVALUATION_END_S
        )
    )

    pinn_weights[pinn_active_mask] = (
        PINN_CONFIG["active_data_weight"]
    )

    pinn_weight_tensor = torch.tensor(
        pinn_weights[:, None],
        device=DEVICE,
    )

    collocation_times = np.linspace(
        0.0,
        float(train_window["model_time_s"][-1]),
        PINN_CONFIG["n_collocation"],
    )

    collocation_times = np.unique(
        np.concatenate(
            [
                collocation_times,
                train_window["model_time_s"],
            ]
        )
    )

    for boundary in [
        PINN_STIMULUS_ONSET,
        PINN_STIMULUS_END,
    ]:
        collocation_times = collocation_times[
            np.abs(collocation_times - boundary) > 0.05
        ]

    collocation_tensor = torch.tensor(
        collocation_times[:, None],
        device=DEVICE,
    )

    PINN_TERMINAL_TIME = float(
        train_window["model_time_s"][-1]
    )


def stimulus_torch(
    time_tensor: torch.Tensor,
):
    return (
        (
            time_tensor >= PINN_STIMULUS_ONSET
        )
        & (
            time_tensor < PINN_STIMULUS_END
        )
    ).to(time_tensor.dtype)


def time_derivative(
    output: torch.Tensor,
    time_tensor: torch.Tensor,
):
    return torch.autograd.grad(
        outputs=output,
        inputs=time_tensor,
        grad_outputs=torch.ones_like(output),
        create_graph=True,
        retain_graph=True,
    )[0]


def physical_residuals(
    model: RealWindowPINN,
    time_tensor: torch.Tensor,
):
    times = (
        time_tensor
        .detach()
        .clone()
        .requires_grad_(True)
    )

    s, f, v, q = model.states(times)

    ds_dt = time_derivative(s, times)
    df_dt = time_derivative(f, times)
    dv_dt = time_derivative(v, times)
    dq_dt = time_derivative(q, times)

    parameters = model.parameter_values()

    epsilon = parameters["epsilon"]
    tau = parameters["tau"]
    alpha = parameters["alpha"]

    kappa_s = 0.65
    kappa_f = 0.41
    E0 = 0.34

    neural_input = stimulus_torch(times)

    extraction = (
        1.0
        - torch.pow(
            torch.tensor(
                1.0 - E0,
                device=DEVICE,
            ),
            1.0 / f,
        )
    )

    return {
        "s": (
            ds_dt
            - (
                epsilon * neural_input
                - kappa_s * s
                - kappa_f * (f - 1.0)
            )
        ),
        "f": df_dt - s,
        "v": (
            tau * dv_dt
            - (
                f
                - torch.pow(
                    v,
                    1.0 / alpha,
                )
            )
        ),
        "q": (
            tau * dq_dt
            - (
                f * extraction / E0
                - torch.pow(
                    v,
                    1.0 / alpha,
                )
                * q / v
            )
        ),
    }


def calcular_perdidas_pinn(
    model: RealWindowPINN,
    lambda_physics: float,
):
    predicted_bold = model.bold(
        pinn_observed_time
    )

    squared_error = (
        (
            predicted_bold
            - pinn_observed_bold
        )
        / pinn_data_scale
    ) ** 2

    data_loss = torch.sum(
        pinn_weight_tensor * squared_error
    ) / torch.sum(pinn_weight_tensor)

    residuals = physical_residuals(
        model,
        collocation_tensor,
    )

    physics_loss = sum(
        torch.mean(
            (
                residual
                / PINN_CONFIG["residual_scale"]
            ) ** 2
        )
        for residual in residuals.values()
    )

    initial_time = torch.zeros(
        (1, 1),
        device=DEVICE,
    )

    terminal_time = torch.full(
        (1, 1),
        PINN_TERMINAL_TIME,
        device=DEVICE,
    )

    s0, f0, v0, q0 = model.states(initial_time)
    sf, ff, vf, qf = model.states(terminal_time)

    initial_loss = (
        torch.mean(s0**2)
        + torch.mean((f0 - 1.0) ** 2)
        + torch.mean((v0 - 1.0) ** 2)
        + torch.mean((q0 - 1.0) ** 2)
    )

    terminal_loss = (
        torch.mean(sf**2)
        + torch.mean((ff - 1.0) ** 2)
        + torch.mean((vf - 1.0) ** 2)
        + torch.mean((qf - 1.0) ** 2)
    )

    total_loss = (
        data_loss
        + lambda_physics * physics_loss
        + PINN_CONFIG["lambda_initial"]
        * initial_loss
        + PINN_CONFIG["lambda_terminal"]
        * terminal_loss
    )

    return {
        "total": total_loss,
        "data": data_loss,
        "physics": physics_loss,
        "initial": initial_loss,
        "terminal": terminal_loss,
    }

In [ ]:
def entrenar_fase_pinn(
    model,
    optimizer,
    n_epochs,
    phase_name,
    lambda_physics,
):
    history = []

    best_loss = np.inf
    best_state = None

    for epoch in range(1, n_epochs + 1):
        model.train()
        optimizer.zero_grad()

        losses = calcular_perdidas_pinn(
            model,
            lambda_physics,
        )

        if not torch.isfinite(losses["total"]):
            raise RuntimeError(
                "La pérdida de la PINN dejó de ser finita."
            )

        losses["total"].backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0,
        )

        optimizer.step()

        total_value = float(
            losses["total"].detach().cpu()
        )

        parameter_values = {
            name: float(value.detach().cpu())
            for name, value in model.parameter_values().items()
        }

        history.append(
            {
                "phase": phase_name,
                "epoch": epoch,
                "total_loss": total_value,
                "data_loss": float(
                    losses["data"].detach().cpu()
                ),
                "physics_loss": float(
                    losses["physics"].detach().cpu()
                ),
                "initial_loss": float(
                    losses["initial"].detach().cpu()
                ),
                "terminal_loss": float(
                    losses["terminal"].detach().cpu()
                ),
                **parameter_values,
            }
        )

        if total_value < best_loss:
            best_loss = total_value
            best_state = copy.deepcopy(
                model.state_dict()
            )

    if best_state is None:
        raise RuntimeError(
            "La PINN no produjo un estado válido."
        )

    model.load_state_dict(best_state)

    return pd.DataFrame(history)


def entrenar_pinn_real(
    train_window: dict,
    random_seed: int,
) -> dict:
    establecer_semilla(random_seed)
    preparar_contexto_pinn(train_window)

    model = RealWindowPINN(
        maximum_time_s=float(
            train_window["model_time_s"][-1]
        )
    ).to(DEVICE)

    start_time = time.time()

    model.epsilon.raw.requires_grad_(False)
    model.tau.raw.requires_grad_(False)
    model.alpha.raw.requires_grad_(False)

    optimizer_phase_1 = torch.optim.Adam(
        model.network.parameters(),
        lr=PINN_CONFIG[
            "phase_1_learning_rate"
        ],
    )

    history_phase_1 = entrenar_fase_pinn(
        model=model,
        optimizer=optimizer_phase_1,
        n_epochs=PINN_CONFIG[
            "phase_1_epochs"
        ],
        phase_name="network_warmup",
        lambda_physics=PINN_CONFIG[
            "lambda_physics_phase_1"
        ],
    )

    model.epsilon.raw.requires_grad_(True)
    model.tau.raw.requires_grad_(True)
    model.alpha.raw.requires_grad_(True)

    optimizer_phase_2 = torch.optim.Adam(
        [
            {
                "params": model.network.parameters(),
                "lr": PINN_CONFIG[
                    "phase_2_network_learning_rate"
                ],
            },
            {
                "params": [
                    model.epsilon.raw,
                    model.tau.raw,
                    model.alpha.raw,
                ],
                "lr": PINN_CONFIG[
                    "phase_2_parameter_learning_rate"
                ],
            },
        ]
    )

    history_phase_2 = entrenar_fase_pinn(
        model=model,
        optimizer=optimizer_phase_2,
        n_epochs=PINN_CONFIG[
            "phase_2_epochs"
        ],
        phase_name="inverse_problem",
        lambda_physics=PINN_CONFIG[
            "lambda_physics_phase_2"
        ],
    )

    lbfgs = torch.optim.LBFGS(
        model.parameters(),
        lr=0.5,
        max_iter=PINN_CONFIG[
            "lbfgs_max_iterations"
        ],
        history_size=50,
        line_search_fn="strong_wolfe",
    )

    lbfgs_log = []
    lbfgs_counter = {"count": 0}

    def closure():
        lbfgs.zero_grad()

        losses = calcular_perdidas_pinn(
            model,
            PINN_CONFIG[
                "lambda_physics_phase_2"
            ],
        )

        losses["total"].backward()
        lbfgs_counter["count"] += 1

        if (
            lbfgs_counter["count"] == 1
            or lbfgs_counter["count"] % 25 == 0
        ):
            parameters = {
                name: float(value.detach().cpu())
                for name, value in model.parameter_values().items()
            }

            lbfgs_log.append(
                {
                    "phase": "lbfgs",
                    "epoch": lbfgs_counter["count"],
                    "total_loss": float(
                        losses["total"].detach().cpu()
                    ),
                    "data_loss": float(
                        losses["data"].detach().cpu()
                    ),
                    "physics_loss": float(
                        losses["physics"].detach().cpu()
                    ),
                    "initial_loss": float(
                        losses["initial"].detach().cpu()
                    ),
                    "terminal_loss": float(
                        losses["terminal"].detach().cpu()
                    ),
                    **parameters,
                }
            )

        return losses["total"]

    _ = lbfgs.step(closure)

    parameters = {
        name: float(value.detach().cpu())
        for name, value in model.parameter_values().items()
    }

    history = pd.concat(
        [
            history_phase_1,
            history_phase_2,
            pd.DataFrame(lbfgs_log),
        ],
        ignore_index=True,
    )

    elapsed_minutes = (
        time.time() - start_time
    ) / 60.0

    return {
        "model": model,
        "parameters": parameters,
        "history": history,
        "elapsed_minutes": elapsed_minutes,
        "lbfgs_iterations": int(
            lbfgs_counter["count"]
        ),
    }

In [ ]:
def simular_ode_real(
    model_times: np.ndarray,
    parameters: dict,
    stimulus_onset_s: float,
    stimulus_duration_s: float,
) -> dict:
    kappa_s = 0.65
    kappa_f = 0.41
    E0 = 0.34
    V0 = 0.02

    k1 = 7.0 * E0
    k2 = 2.0
    k3 = 2.0 * E0 - 0.2

    def stimulus(current_time):
        return float(
            (
                current_time >= stimulus_onset_s
            )
            and (
                current_time
                < stimulus_onset_s
                + stimulus_duration_s
            )
        )

    def rhs(current_time, state):
        s, f, v, q = state

        f_safe = max(float(f), 1e-8)
        v_safe = max(float(v), 1e-8)

        extraction = (
            1.0
            - (1.0 - E0) ** (
                1.0 / f_safe
            )
        )

        neural_input = stimulus(current_time)

        return [
            (
                parameters["epsilon"]
                * neural_input
                - kappa_s * s
                - kappa_f * (f - 1.0)
            ),
            s,
            (
                f
                - v_safe ** (
                    1.0
                    / parameters["alpha"]
                )
            ) / parameters["tau"],
            (
                f * extraction / E0
                - v_safe ** (
                    1.0
                    / parameters["alpha"]
                )
                * q / v_safe
            ) / parameters["tau"],
        ]

    model_times = np.asarray(
        model_times,
        dtype=float,
    )

    solution = solve_ivp(
        fun=rhs,
        t_span=(
            float(model_times[0]),
            float(model_times[-1]),
        ),
        y0=[0.0, 1.0, 1.0, 1.0],
        t_eval=model_times,
        max_step=0.02,
        rtol=1e-9,
        atol=1e-11,
    )

    if not solution.success:
        raise RuntimeError(solution.message)

    s, f, v, q = solution.y

    bold = V0 * (
        k1 * (1.0 - q)
        + k2 * (1.0 - q / v)
        + k3 * (1.0 - v)
    )

    return {
        "bold_fraction": bold,
        "s": s,
        "f": f,
        "v": v,
        "q": q,
    }


def estimar_hrf_real(
    parameters: dict,
) -> pd.DataFrame:
    hrf_dt_s = 0.05
    event_onset_s = 5.0
    event_duration_s = 1.0
    final_relative_time_s = 32.0

    model_times = np.arange(
        0.0,
        event_onset_s
        + final_relative_time_s
        + hrf_dt_s,
        hrf_dt_s,
    )

    simulation = simular_ode_real(
        model_times=model_times,
        parameters=parameters,
        stimulus_onset_s=event_onset_s,
        stimulus_duration_s=event_duration_s,
    )

    relative_times = (
        model_times - event_onset_s
    )

    mask = relative_times >= 0.0

    return pd.DataFrame(
        {
            "time_from_onset_s": relative_times[mask],
            "hrf_fraction": simulation[
                "bold_fraction"
            ][mask],
        }
    )

In [ ]:
def evaluar_prediccion(
    window: dict,
    prediction: np.ndarray,
) -> dict:
    evaluation_mask = window["evaluation_mask"]

    return calcular_metricas(
        window["response_fraction"][
            evaluation_mask
        ],
        prediction[evaluation_mask],
    )


def ejecutar_experimento_real(
    experiment: pd.Series,
) -> dict:
    scenario_name = str(
        experiment["scenario"]
    )

    train_block = int(
        experiment["train_block"]
    )

    test_block = int(
        experiment["test_block"]
    )

    random_seed = int(
        experiment["experiment_seed"]
    )

    context = cargar_contexto_experimento(
        scenario_name=scenario_name,
        train_block=train_block,
        test_block=test_block,
    )

    train_window = context["train_window"]
    test_window = context["test_window"]

    glm_result = ejecutar_glm(
        train_window=train_window,
        test_window=test_window,
    )

    mlp_result = ejecutar_mlp(
        train_window=train_window,
        test_window=test_window,
        tr_s=context["tr_s"],
        random_seed=random_seed + 1,
    )

    pinn_result = entrenar_pinn_real(
        train_window=train_window,
        random_seed=random_seed + 2,
    )

    pinn_train_simulation = simular_ode_real(
        model_times=train_window["model_time_s"],
        parameters=pinn_result["parameters"],
        stimulus_onset_s=SECONDS_BEFORE,
        stimulus_duration_s=float(
            train_window["duration_s"]
        ),
    )

    pinn_test_simulation = simular_ode_real(
        model_times=test_window["model_time_s"],
        parameters=pinn_result["parameters"],
        stimulus_onset_s=SECONDS_BEFORE,
        stimulus_duration_s=float(
            test_window["duration_s"]
        ),
    )

    model_outputs = {
        "GLM": {
            "train_prediction": glm_result[
                "train_prediction"
            ],
            "test_prediction": glm_result[
                "test_prediction"
            ],
        },
        "MLP": {
            "train_prediction": mlp_result[
                "train_prediction"
            ],
            "test_prediction": mlp_result[
                "test_prediction"
            ],
        },
        "PINN": {
            "train_prediction": pinn_train_simulation[
                "bold_fraction"
            ],
            "test_prediction": pinn_test_simulation[
                "bold_fraction"
            ],
        },
    }

    metric_rows = []

    for model_name, output in model_outputs.items():
        train_metrics = evaluar_prediccion(
            train_window,
            output["train_prediction"],
        )

        test_metrics = evaluar_prediccion(
            test_window,
            output["test_prediction"],
        )

        metric_rows.append(
            {
                "scenario": scenario_name,
                "run": experiment["run"],
                "roi": experiment["roi"],
                "target_condition": experiment[
                    "target_condition"
                ],
                "train_block": train_block,
                "test_block": test_block,
                "model": model_name,
                "random_seed": random_seed,
                "train_rmse_percent": train_metrics[
                    "rmse_percent"
                ],
                "train_mae_percent": train_metrics[
                    "mae_percent"
                ],
                "train_r2": train_metrics["r2"],
                "train_pearson_r": train_metrics[
                    "pearson_r"
                ],
                "test_rmse_percent": test_metrics[
                    "rmse_percent"
                ],
                "test_mae_percent": test_metrics[
                    "mae_percent"
                ],
                "test_r2": test_metrics["r2"],
                "test_pearson_r": test_metrics[
                    "pearson_r"
                ],
            }
        )

    parameter_row = {
        "scenario": scenario_name,
        "run": experiment["run"],
        "roi": experiment["roi"],
        "target_condition": experiment[
            "target_condition"
        ],
        "train_block": train_block,
        "test_block": test_block,
        "epsilon": pinn_result[
            "parameters"
        ]["epsilon"],
        "tau": pinn_result[
            "parameters"
        ]["tau"],
        "alpha": pinn_result[
            "parameters"
        ]["alpha"],
        "pinn_elapsed_minutes": pinn_result[
            "elapsed_minutes"
        ],
        "lbfgs_iterations": pinn_result[
            "lbfgs_iterations"
        ],
    }

    for parameter_name, (
        lower_bound,
        upper_bound,
    ) in PARAMETER_BOUNDS.items():
        estimated_value = pinn_result[
            "parameters"
        ][parameter_name]

        normalized_position = (
            estimated_value - lower_bound
        ) / (
            upper_bound - lower_bound
        )

        parameter_row[
            f"{parameter_name}_normalized_position"
        ] = float(normalized_position)

        parameter_row[
            f"{parameter_name}_near_bound"
        ] = bool(
            normalized_position < 0.05
            or normalized_position > 0.95
        )

    prediction_table = pd.DataFrame(
        {
            "scenario": scenario_name,
            "train_block": train_block,
            "test_block": test_block,
            "relative_time_s": test_window[
                "relative_time_s"
            ],
            "model_time_s": test_window[
                "model_time_s"
            ],
            "stimulus": test_window[
                "stimulus"
            ],
            "observed_fraction": test_window[
                "response_fraction"
            ],
            "glm_fraction": glm_result[
                "test_prediction"
            ],
            "mlp_fraction": mlp_result[
                "test_prediction"
            ],
            "pinn_fraction": pinn_test_simulation[
                "bold_fraction"
            ],
            "evaluation_mask": test_window[
                "evaluation_mask"
            ].astype(int),
        }
    )

    hrf_table = estimar_hrf_real(
        pinn_result["parameters"]
    )

    return {
        "context": context,
        "metric_rows": metric_rows,
        "parameter_row": parameter_row,
        "prediction_table": prediction_table,
        "mlp_result": mlp_result,
        "pinn_result": pinn_result,
        "hrf_table": hrf_table,
    }

In [ ]:
def leer_csv_si_existe(
    path: Path,
) -> pd.DataFrame:
    if path.exists():
        return pd.read_csv(path)

    return pd.DataFrame()


metrics_table = leer_csv_si_existe(
    METRICS_PATH
)

parameters_table = leer_csv_si_existe(
    PARAMETERS_PATH
)

status_table = leer_csv_si_existe(
    STATUS_PATH
)

failures_table = leer_csv_si_existe(
    FAILURES_PATH
)

completed_keys = set()

if not status_table.empty:
    completed_keys = set(
        zip(
            status_table["scenario"],
            status_table[
                "train_block"
            ].astype(int),
            status_table[
                "test_block"
            ].astype(int),
        )
    )

print(
    "Experimentos ya terminados:",
    len(completed_keys),
)

In [ ]:
for experiment_index, (_, experiment) in enumerate(
    experiment_table.iterrows(),
    start=1,
):
    key = (
        str(experiment["scenario"]),
        int(experiment["train_block"]),
        int(experiment["test_block"]),
    )

    if key in completed_keys:
        print(
            f"Omitido {key}: ya está guardado."
        )
        continue

    print("\n" + "=" * 80)
    print(
        f"Experimento {experiment_index}/"
        f"{len(experiment_table)}: {key}"
    )

    experiment_start = time.time()

    try:
        result = ejecutar_experimento_real(
            experiment
        )

        metrics_table = pd.concat(
            [
                metrics_table,
                pd.DataFrame(
                    result["metric_rows"]
                ),
            ],
            ignore_index=True,
        )

        parameters_table = pd.concat(
            [
                parameters_table,
                pd.DataFrame(
                    [result["parameter_row"]]
                ),
            ],
            ignore_index=True,
        )

        file_key = (
            f"{key[0]}_"
            f"b{key[1]}_to_b{key[2]}"
        )

        result["prediction_table"].to_csv(
            PREDICTIONS_DIR
            / f"{file_key}.csv",
            index=False,
        )

        result["mlp_result"][
            "training_result"
        ]["history"].to_csv(
            HISTORIES_DIR
            / f"{file_key}_mlp.csv",
            index=False,
        )

        result["pinn_result"][
            "history"
        ].to_csv(
            HISTORIES_DIR
            / f"{file_key}_pinn.csv",
            index=False,
        )

        result["hrf_table"].to_csv(
            HRF_DIR
            / f"{file_key}.csv",
            index=False,
        )

        torch.save(
            {
                "model_state_dict": (
                    result["mlp_result"][
                        "training_result"
                    ]["model"].state_dict()
                ),
                "x_mean": result[
                    "mlp_result"
                ]["training_result"]["x_mean"],
                "x_std": result[
                    "mlp_result"
                ]["training_result"]["x_std"],
                "y_mean": result[
                    "mlp_result"
                ]["training_result"]["y_mean"],
                "y_std": result[
                    "mlp_result"
                ]["training_result"]["y_std"],
                "n_lags": result[
                    "mlp_result"
                ]["n_lags"],
            },
            MODELS_DIR
            / f"{file_key}_mlp.pt",
        )

        torch.save(
            {
                "model_state_dict": (
                    result["pinn_result"][
                        "model"
                    ].state_dict()
                ),
                "parameters": result[
                    "pinn_result"
                ]["parameters"],
                "configuration": PINN_CONFIG,
            },
            MODELS_DIR
            / f"{file_key}_pinn.pt",
        )

        predictions = result[
            "prediction_table"
        ]

        plt.figure(figsize=(11, 6))

        plt.plot(
            predictions["relative_time_s"],
            100.0
            * predictions[
                "observed_fraction"
            ],
            linewidth=2.0,
            label="BOLD observada",
        )

        plt.plot(
            predictions["relative_time_s"],
            100.0
            * predictions[
                "glm_fraction"
            ],
            linewidth=1.7,
            label="GLM",
        )

        plt.plot(
            predictions["relative_time_s"],
            100.0
            * predictions[
                "mlp_fraction"
            ],
            linewidth=1.7,
            label="MLP",
        )

        plt.plot(
            predictions["relative_time_s"],
            100.0
            * predictions[
                "pinn_fraction"
            ],
            linewidth=2.0,
            label="PINN + ODE",
        )

        duration_s = float(
            result["context"][
                "test_window"
            ]["duration_s"]
        )

        plt.axvspan(
            0.0,
            duration_s,
            alpha=0.15,
            label="Movimiento de mano",
        )

        plt.axvline(
            EVALUATION_END_S,
            linestyle="--",
            linewidth=1.0,
            label="Fin de evaluación",
        )

        plt.axhline(
            0.0,
            linewidth=0.8,
        )

        plt.xlabel(
            "Tiempo respecto del inicio del bloque [s]"
        )
        plt.ylabel(
            "Cambio BOLD corregido [%]"
        )

        plt.title(
            "Predicción fuera de muestra sobre datos reales\n"
            f"{key[0]}: bloque {key[1]} → bloque {key[2]}"
        )

        plt.legend()
        plt.grid(alpha=0.25)
        plt.tight_layout()

        plt.savefig(
            FIGURES_DIR
            / f"real_models_{file_key}.png",
            dpi=300,
            bbox_inches="tight",
        )

        plt.close()

        elapsed_minutes = (
            time.time() - experiment_start
        ) / 60.0

        status_row = {
            "scenario": key[0],
            "train_block": key[1],
            "test_block": key[2],
            "status": "completed",
            "elapsed_minutes_total": (
                elapsed_minutes
            ),
        }

        status_table = pd.concat(
            [
                status_table,
                pd.DataFrame(
                    [status_row]
                ),
            ],
            ignore_index=True,
        )

        metrics_table.to_csv(
            METRICS_PATH,
            index=False,
        )

        parameters_table.to_csv(
            PARAMETERS_PATH,
            index=False,
        )

        status_table.to_csv(
            STATUS_PATH,
            index=False,
        )

        completed_keys.add(key)

        experiment_metrics = (
            metrics_table.loc[
                (
                    metrics_table["scenario"]
                    == key[0]
                )
                & (
                    metrics_table[
                        "train_block"
                    ] == key[1]
                )
                & (
                    metrics_table[
                        "test_block"
                    ] == key[2]
                )
            ]
        )

        print(
            experiment_metrics[
                [
                    "model",
                    "test_rmse_percent",
                    "test_r2",
                    "test_pearson_r",
                ]
            ].to_string(index=False)
        )

        print(
            f"Terminado en "
            f"{elapsed_minutes:.2f} min."
        )

        del result

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as error:
        print(
            f"ERROR en {key}: {error}"
        )

        failure_row = {
            "scenario": key[0],
            "train_block": key[1],
            "test_block": key[2],
            "error_type": type(error).__name__,
            "error_message": str(error),
            "traceback": traceback.format_exc(),
        }

        failures_table = pd.concat(
            [
                failures_table,
                pd.DataFrame(
                    [failure_row]
                ),
            ],
            ignore_index=True,
        )

        failures_table.to_csv(
            FAILURES_PATH,
            index=False,
        )

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [ ]:
metrics_table = pd.read_csv(
    METRICS_PATH
)

parameters_table = pd.read_csv(
    PARAMETERS_PATH
)

status_table = pd.read_csv(
    STATUS_PATH
)

print("Experimentos completados:", len(status_table))
print("Filas de métricas:", len(metrics_table))
print("Estimaciones PINN:", len(parameters_table))

display(
    metrics_table.sort_values(
        [
            "scenario",
            "train_block",
            "model",
        ]
    )
)

In [ ]:
real_model_summary = (
    metrics_table
    .groupby(
        "model",
        as_index=False,
    )
    .agg(
        n_experiments=(
            "test_rmse_percent",
            "size",
        ),
        test_rmse_mean=(
            "test_rmse_percent",
            "mean",
        ),
        test_rmse_std=(
            "test_rmse_percent",
            "std",
        ),
        test_rmse_median=(
            "test_rmse_percent",
            "median",
        ),
        test_mae_mean=(
            "test_mae_percent",
            "mean",
        ),
        test_r2_mean=(
            "test_r2",
            "mean",
        ),
        test_r2_std=(
            "test_r2",
            "std",
        ),
        test_pearson_mean=(
            "test_pearson_r",
            "mean",
        ),
        train_rmse_mean=(
            "train_rmse_percent",
            "mean",
        ),
    )
    .sort_values(
        "test_rmse_mean"
    )
)

real_model_summary.to_csv(
    RESULTS_DIR
    / "real_model_summary.csv",
    index=False,
)

display(real_model_summary)

In [ ]:
wide_rmse = metrics_table.pivot(
    index=[
        "scenario",
        "train_block",
        "test_block",
    ],
    columns="model",
    values="test_rmse_percent",
)

winner_table = (
    wide_rmse
    .idxmin(axis=1)
    .value_counts()
    .rename_axis("model")
    .reset_index(name="n_wins")
)

winner_table.to_csv(
    RESULTS_DIR
    / "real_model_wins.csv",
    index=False,
)

display(winner_table)

In [ ]:
parameter_consistency_rows = []

for scenario_name, group in parameters_table.groupby(
    "scenario"
):
    if len(group) != 2:
        continue

    first_direction = group.loc[
        group["train_block"] == 1
    ].iloc[0]

    second_direction = group.loc[
        group["train_block"] == 2
    ].iloc[0]

    row = {
        "scenario": scenario_name,
    }

    for parameter_name in [
        "epsilon",
        "tau",
        "alpha",
    ]:
        first_value = float(
            first_direction[
                parameter_name
            ]
        )

        second_value = float(
            second_direction[
                parameter_name
            ]
        )

        mean_magnitude = (
            abs(first_value)
            + abs(second_value)
        ) / 2.0

        symmetric_difference = (
            np.nan
            if mean_magnitude < 1e-12
            else (
                100.0
                * abs(
                    first_value
                    - second_value
                )
                / mean_magnitude
            )
        )

        row[
            f"{parameter_name}_block1"
        ] = first_value

        row[
            f"{parameter_name}_block2"
        ] = second_value

        row[
            f"{parameter_name}_absolute_difference"
        ] = abs(
            first_value
            - second_value
        )

        row[
            f"{parameter_name}_symmetric_difference_percent"
        ] = symmetric_difference

    parameter_consistency_rows.append(row)

parameter_consistency = pd.DataFrame(
    parameter_consistency_rows
)

parameter_consistency.to_csv(
    RESULTS_DIR
    / "real_pinn_parameter_consistency.csv",
    index=False,
)

display(parameter_consistency)

In [ ]:
hrf_consistency_rows = []

for scenario_name in SCENARIOS:
    first_path = (
        HRF_DIR
        / f"{scenario_name}_b1_to_b2.csv"
    )

    second_path = (
        HRF_DIR
        / f"{scenario_name}_b2_to_b1.csv"
    )

    if (
        not first_path.exists()
        or not second_path.exists()
    ):
        continue

    first_hrf = pd.read_csv(
        first_path
    )

    second_hrf = pd.read_csv(
        second_path
    )

    first_values = first_hrf[
        "hrf_fraction"
    ].to_numpy(dtype=float)

    second_values = second_hrf[
        "hrf_fraction"
    ].to_numpy(dtype=float)

    hrf_consistency_rows.append(
        {
            "scenario": scenario_name,
            "hrf_rmse_percent": float(
                100.0
                * np.sqrt(
                    mean_squared_error(
                        first_values,
                        second_values,
                    )
                )
            ),
            "hrf_pearson_r": (
                correlacion_segura(
                    first_values,
                    second_values,
                )
            ),
            "peak_block1_percent": float(
                100.0
                * np.max(
                    first_values
                )
            ),
            "peak_block2_percent": float(
                100.0
                * np.max(
                    second_values
                )
            ),
        }
    )

hrf_consistency = pd.DataFrame(
    hrf_consistency_rows
)

hrf_consistency.to_csv(
    RESULTS_DIR
    / "real_pinn_hrf_consistency.csv",
    index=False,
)

display(hrf_consistency)

In [ ]:
plot_table = (
    metrics_table
    .assign(
        experiment_label=lambda table: (
            table["scenario"]
            + "\nB"
            + table["train_block"].astype(str)
            + "→B"
            + table["test_block"].astype(str)
        )
    )
)

experiment_order = (
    plot_table[
        [
            "scenario",
            "train_block",
            "test_block",
            "experiment_label",
        ]
    ]
    .drop_duplicates()
    ["experiment_label"]
    .tolist()
)

model_order = [
    "GLM",
    "MLP",
    "PINN",
]

x = np.arange(
    len(experiment_order)
)

width = 0.25

plt.figure(figsize=(15, 6))

for model_index, model_name in enumerate(
    model_order
):
    model_values = (
        plot_table.loc[
            plot_table["model"]
            == model_name
        ]
        .set_index(
            "experiment_label"
        )
        .reindex(
            experiment_order
        )[
            "test_rmse_percent"
        ]
        .to_numpy(dtype=float)
    )

    plt.bar(
        x
        + (
            model_index - 1
        )
        * width,
        model_values,
        width=width,
        label=model_name,
    )

plt.xticks(
    x,
    experiment_order,
    rotation=35,
    ha="right",
)

plt.ylabel(
    "RMSE fuera de muestra "
    "[puntos porcentuales BOLD]"
)

plt.title(
    "Comparación de modelos sobre datos fMRI reales"
)

plt.legend()
plt.grid(
    axis="y",
    alpha=0.25,
)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR
    / "real_models_rmse_by_experiment.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()